In [ ]:
import math
import torch 
import torch.nn as nn
import pandas as pd 
import numpy as np
import PIL
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torch.nn import functional as F
from torch.autograd import Variable
from glob import glob
from collections import OrderedDict
from pretrainedmodels import se_resnext50_32x4d, se_resnext101_32x4d

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

In [ ]:
class ResNext50(nn.Module):

    def __init__(self, freeze):
        super().__init__()
        self.model = se_resnext50_32x4d(num_classes=1000)
        self.model.avg_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Dropout(0.5))
        self.model.last_linear = nn.Linear(2048, 1103, bias=True)
        self.__init_weights__(self.model.last_linear)
        if freeze:
            for name, param in self.model.named_parameters():
                if 'last_linear' not in name:
                    param.requires_grad = False
        return None

    def __init_weights__(self, layer):
        if type(layer) == nn.Linear:
            nn.init.kaiming_normal_(layer.weight)
        return None

    def forward(self, image):
        output = self.model(image)
        return output

class ResNext100(nn.Module):

    def __init__(self, freeze):
        super().__init__()
        self.model = se_resnext101_32x4d(num_classes=1000)
        self.model.avg_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Dropout(0.5))
        self.model.last_linear = nn.Linear(2048, 1103, bias=True)
        self.__init_weights__(self.model.last_linear)
        if freeze:
            for name, param in self.model.named_parameters():
                if 'last_linear' not in name:
                    param.requires_grad = False
        return None

    def __init_weights__(self, layer):
        if type(layer) == nn.Linear:
            nn.init.kaiming_normal_(layer.weight)
        return None

    def forward(self, image):
        output = self.model(image)
        return output

In [ ]:
class ImageDataset(Dataset):

    def __init__(self, path, version):
        self.path = path
        self.transform = self.__transform_v1__() if version == 'v1' else self.__transform_v2__()
        self.image = pd.read_csv('../input/imet-2019-fgvc6/sample_submission.csv')
        self.image = self.image['id'].tolist()
        return None

    def __transform_v1__(self):
        process = []
        process.append(transforms.RandomHorizontalFlip(p=0.5))
        process.append(transforms.Resize(size=(256,256)))
        process.append(transforms.ToTensor())
        process.append(transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]))
        process = transforms.Compose(process)
        return process

    def __transform_v2__(self):
        process = []
        process.append(transforms.RandomHorizontalFlip(p=0.5))
        process.append(transforms.RandomCrop(size=(300,300), pad_if_needed=True))
        process.append(transforms.ToTensor())
        process.append(transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]))
        process = transforms.Compose(process)
        return process

    def __len__(self):
        return len(self.image)

    def __image__(self, image_idx):
        image = Image.open(self.path + '/' + image_idx + '.png')
        image = self.transform(image)
        return image
    
    def __getitem__(self, idx):
        image = self.__image__(self.image[idx])
        sample = {'idx': self.image[idx], 'image': image}
        return sample

In [ ]:
def scoreLoader(image_path, version, batch):
    score = ImageDataset(image_path, version)
    total = len(score)
    print('Score Images:', total)
    if total % batch == 1: batch += 1 
    score = DataLoader(score, batch_size=batch, shuffle=False, num_workers=2, drop_last=False)
    return score

def loadModel(model, path):
    results = torch.load(path, map_location=DEVICE)
    model.load_state_dict(results['model_state_dict'])
    loss = results['loss']
    print('Model Loaded:', 'Loss:', loss)
    return model, loss

def scoreModel(model, data, load_path, save_path):
    model.eval()
    model, _ = loadModel(model, load_path)
    idx = []
    scores = []
    for sample in data:
        idx += sample['idx']
        image = Variable(sample['image'].float().to(DEVICE))
        preds = torch.sigmoid(model(image))
        scores += [preds.cpu().data.numpy()]
    del image, preds
    torch.cuda.empty_cache()
    scores = np.vstack(scores)
    index = np.vstack(idx)
    index = pd.DataFrame(index, columns=['id'])
    scores = pd.DataFrame(np.around(np.array(scores), decimals=5))
    scores.columns = ['scr_' + str(x) for x in range(scores.shape[1])]
    scores = index.join(scores)
    scores.to_csv(save_path, index=False, compression='gzip')
    print('Score Data:', scores.shape)
    return None

def executeScore(name, version, load_path, save_path):
    loader = {}
    loader['image_path'] = '../input/imet-2019-fgvc6/test/'
    loader['version'] = version
    loader['batch'] = 16
    score_data = scoreLoader(**loader)
    if name == 'resnext50':
        model = ResNext50(freeze=False).to(DEVICE)
    else:
        model = ResNext100(freeze=False).to(DEVICE)
    trainer = {}
    trainer['model'] = model
    trainer['data'] = score_data
    trainer['load_path'] = '../input/imetmodel/' + load_path
    trainer['save_path'] = '../working/' + save_path
    scoreModel(**trainer)
    model = model.cpu()
    del model
    torch.cuda.empty_cache()
    return None

In [ ]:
executeScore('resnext100','v2','resnext100_v2/stage_2_1.pt','score_11.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_2.pt','score_12.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_3.pt','score_13.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_4.pt','score_14.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_5.pt','score_15.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_6.pt','score_16.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_7.pt','score_17.csv.gz')
executeScore('resnext100','v2','resnext100_v2/stage_2_8.pt','score_18.csv.gz')

In [ ]:
executeScore('resnext50','v1','resnext50_v1/stage_2_1.pt','score_1.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_2.pt','score_2.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_3.pt','score_3.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_4.pt','score_4.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_5.pt','score_5.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_6.pt','score_6.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_7.pt','score_7.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_8.pt','score_8.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_9.pt','score_9.csv.gz')
executeScore('resnext50','v1','resnext50_v1/stage_2_10.pt','score_10.csv.gz')

In [ ]:
version_1 = pd.DataFrame([])
for idx in range(1,11):
    temp = None
    temp = pd.read_csv('../working/score_{}.csv.gz'.format(idx), compression='gzip')
    temp.iloc[:,1:] = temp.iloc[:,1:].astype(np.float16)
    version_1 = version_1.append(temp)
    del temp
version_1 = version_1.groupby('id').mean().reset_index()

version_2 = pd.DataFrame([])
for idx in range(11,19):
    temp = None
    temp = pd.read_csv('../working/score_{}.csv.gz'.format(idx), compression='gzip')
    temp.iloc[:,1:] = temp.iloc[:,1:].astype(np.float16)
    version_2 = version_2.append(temp)
    del temp
version_2 = version_2.groupby('id').mean().reset_index()

In [ ]:
version_1.iloc[:,1:] = version_1.iloc[:,1:] * 0.7
version_2.iloc[:,1:] = version_2.iloc[:,1:] * 0.3

In [ ]:
data = version_1.append(version_2)
data = data.groupby('id').sum().reset_index()
del version_1, version_2
version_1 = None
version_2 = None 

In [ ]:
def processLabel(row):
    labels = []
    row = list(row)
    candidate = np.argsort(row)[-10:]
    for label in candidate:
        if row[label] > 0.2: 
            labels += [str(label)]
    if len(labels) == 0:
        labels += [str(np.argmax(row))]
    labels = ' '.join(labels)
    return labels

In [ ]:
submit = data[['id']].copy()
submit['attribute_ids'] = data.iloc[:,1:].apply(lambda x : processLabel(x), axis=1)
sample = pd.read_csv('../input/imet-2019-fgvc6/sample_submission.csv')
submit = submit.append(sample)
submit = submit.drop_duplicates(subset=['id'], keep='first')
print('# Submit:', submit.shape)
submit.to_csv('submission.csv', index=False)